# S3Retriever Usage Guide

This notebook demonstrates how to use the `S3Retriever` class and the `iter_embeddings` iterator to load embeddings across different algorithms, dataset sizes, and quality levels.

In [ ]:
import os

import pandas as pd
import numpy as np

from scaling_laws.s3_retriever import S3Retriever, EmbeddingResult

## 1. Initializing the retriever

Point at a local data directory or use the default public S3 bucket.

In [ ]:
# Local copy
data = S3Retriever("/home/igor/noise_scaling/data")

# Or from S3 (no credentials needed):
# data = S3Retriever()  # defaults to s3://measurement-noise-scaling-laws/data/

## 2. Discovering what is available

In [ ]:
print("Datasets:", data.list_datasets())
print("Algorithms:", data.list_algorithms())
print()
for ds in data.list_datasets():
    print(f"--- {ds} ---")
    print(f"  Sizes:    {data.list_sizes(ds)}")
    print(f"  Qualities:{data.list_qualities(ds)}")
    print(f"  Signals:  {data.list_signals(ds)}")

## 3. Loading a single embedding

In [ ]:
emb = data.load_embeddings("merfish", num_cells=7113, quality=1.0, algorithm="PCA")
print(f"Shape: {emb.shape}")
emb.head()

## 4. Loading a single MI score

In [ ]:
mi = data.load_mutual_information(
    "merfish", num_cells=7113, quality=1.0,
    algorithm="Geneformer", signal="ng_idx", seed=42,
)
print(f"MI = {mi:.4f} nats")

## 5. Iterating over embeddings with `iter_embeddings`

The iterator yields an `EmbeddingResult` dataclass for every (dataset, size, quality, algorithm) combination. Missing files are silently skipped. A tqdm progress bar shows the current configuration.

In [ ]:
# Compare all algorithms on merfish at a single (size, quality)
for result in data.iter_embeddings(
    datasets=["merfish"],
    sizes={"merfish": [7113]},
    qualities={"merfish": [1.0]},
):
    print(
        f"{result.algorithm:20s}  "
        f"dim={result.embedding_dim:<4d}  "
        f"shape={str(result.embedding.shape):<16s}  "
        f"signals={result.signals}"
    )

### 5a. Filtering to specific algorithms

In [ ]:
for result in data.iter_embeddings(
    datasets=["merfish"],
    sizes={"merfish": [7113]},
    qualities={"merfish": [1.0]},
    algorithms=["PCA", "SCVI"],
):
    print(f"{result.algorithm}: {result.embedding.shape}")

### 5b. Sweeping across sizes for one algorithm

In [ ]:
rows = []
for result in data.iter_embeddings(
    datasets=["merfish"],
    qualities={"merfish": [1.0]},
    algorithms=["PCA"],
):
    rows.append({
        "num_cells": result.num_cells,
        "embedding_dim": result.embedding_dim,
        "mean_norm": np.linalg.norm(result.embedding.values, axis=1).mean(),
    })

df = pd.DataFrame(rows)
df

## 6. Collecting all MI results for a dataset

In [ ]:
mi_df = data.collect_all_mi_results("merfish", algorithms=["PCA", "SCVI"])
print(f"{len(mi_df)} MI results collected")
mi_df.head(10)

## 7. Full multi-dataset sweep

Iterate across multiple datasets and collect summary statistics.

In [ ]:
summary = []
for result in data.iter_embeddings(
    datasets=["merfish", "PBMC"],
    algorithms=["PCA", "Geneformer"],
):
    summary.append({
        "dataset": result.dataset,
        "num_cells": result.num_cells,
        "quality": result.quality,
        "algorithm": result.algorithm,
        "embedding_dim": result.embedding_dim,
        "n_test_cells": result.embedding.shape[0],
    })

summary_df = pd.DataFrame(summary)
print(f"Loaded {len(summary_df)} configurations")
summary_df.head()